# Lab 8

In [10]:
import pandas as pd

listings = pd.read_csv('../listings_clean.csv', parse_dates=['first_review', 'last_review'])

listings.shape

(4574, 73)

## Task 1


In [39]:
#A night in Stockholm costs 1791.

prices = listings['price']

print(f"listings with no price: {prices.isna().sum()}, mean: {prices.isna().mean().round(2)}")
print(f"listings with a price: {prices.notna().sum()}, mean: {prices.notna().mean().round(2)}")

print(f"price mean: {prices.mean().round(1)}, price median: {prices.median()}")

#The price median is 1791 on the 2957 entries with a price (of 4574 entries).

listings with no price: 1617, mean: 0.35
listings with a price: 2957, mean: 0.65
price mean: 2327.6, price median: 1791.0


In [40]:
availability_groups = pd.cut(
    listings['availability_365'],
    bins=[-1, 0, 1, 5, 30, 90, 180, 365],
    labels=['0', '1', '2-5', '6-30', '31-90', '91-180', '181-365'],
    include_lowest=True
)

#Availability for listings with missing price.
availability_groups[listings['price'].isna()].value_counts()

availability_365
0          1490
181-365      55
1            24
2-5          21
31-90        15
6-30          6
91-180        6
Name: count, dtype: int64

In [56]:
minimum_long_all = listings[listings['minimum_nights'] >= 28]

minimum_short_all = listings[listings['minimum_nights'] < 28]

minimum_long_price = listings[(listings['price'].notna()) & (listings['minimum_nights'] >= 28)]

print(f'median (>=28): {minimum_long_all['price'].median().round(1)}')
print(f'median (<28): {minimum_short_all['price'].median().round(1)}, amount of listings: {len(minimum_short_all)}')

#With this, we can also (roughly) see how many are missing with the entire amount above as comparison.
print(f'all listings where the minimum amount of nights is at least 28 days: {len(minimum_long_all)}') 
print(f'all such listings (with prices): {len(minimum_long_price)}')

median (>=28): 708.9
median (<28): 1844.2, amount of listings: 4382
all listings where the minimum amount of nights is at least 28 days: 188
all such listings (with prices): 97


Most entries are not available at all during the entire year (1490 entries).

**Verdict:** Holds with reservations.

A majority of the listings without a price are also not available at all year round (1490/1617), and 
about as many listings with a price have almost equally many with at least 28 as minimum days to stay as those without. 

Listings with an amount of minimum nights below 28 days match up fairly well with the median price for all listings, and these also make up the majority (4382/4574) of the listings.

So there are still a fair amount of values holding back a conclusive verdict, but the values that do exist support the claim to quite some extent. 

## Task 2


claim 2: the listings with a perfect 5.00 rating are the best in stockholm.

values behind these numbers...:

In [ ]:
has_reviews = listings[listings['review_scores_rating'].notna()]

print(f'listings that have at least one review: {len(has_reviews)} entries')

print(f'that is, {round((len(has_reviews) / len(listings) * 100), 1)} percent')

listings that have at least one review: 3930 entries
that is, 85.9 percent


First of all, 85.9 have at least one review, which is a huge majority.

In [80]:
perfectly_reviewed = has_reviews[has_reviews['review_scores_rating'] == 5.00]

not_perfectly_reviewed = has_reviews[has_reviews['review_scores_rating'] != 5.00]

print(f'perfectly rated reviews: {len(perfectly_reviewed)}')

perfectly_reviewed_20_plus = perfectly_reviewed[perfectly_reviewed['number_of_reviews'] >= 20]

print(f'perfectly rated listings with at least 20 reviews: {len(perfectly_reviewed_20_plus)}, {round(len(perfectly_reviewed_20_plus) / len(perfectly_reviewed) * 100, 1)} percent')

perfectly rated reviews: 1484
perfectly rated listings with at least 20 reviews: 130, 8.8 percent


In [81]:
print(f'median for perfectly reviewed ones: {perfectly_reviewed['number_of_reviews'].median()}, median for the others: {not_perfectly_reviewed['number_of_reviews'].median()}')

median for perfectly reviewed ones: 4.0, median for the others: 25.0


**Verdict:** Does not hold.

Not only is the proportion of reviews for perfectly rated listings significantly smaller in comparison to the non-perfect ones (4 vs. 25), but the
amount itself is a little more than a third of all reviewed listings. 

## Task 3


"the market has grown: 131 listings in 2015, but 610 in 2024."

this probably needs that we check on the first_review-year with .dt.year

In [ ]:
first_review_years = listings['first_review'].dt.year
latest_review_2026 = listings['last_review'].dt.year == 2026

latest_review_2026.groupby(first_review_years).agg(['count', 'mean']).loc[2016:].round(2)

#Counts: 131 in 2016, 610 in 2024
#Shares: 53% from 2016 with latest review this year (2026), and 67% on 2024 

,count,mean
first_review,,
2016.0,131,0.53
2017.0,148,0.53
2018.0,191,0.51
2019.0,234,0.55
2020.0,96,0.54
2021.0,159,0.58
2022.0,332,0.52
2023.0,565,0.59
2024.0,610,0.67


**Verdict:** Cannot be confluded from this table.

When a listing started and when the first review came is not (and highly unlikely) happening the same day.
That is the earliest information, and so no other information (which would be something from the current day - being **last_review**) really assists in drawing any conclusion.
It is just a matter about missing a crucial piece.

## Task 4


In [ ]:
#Find all listings with a host
has_host = listings[listings['host_id'].notna()]
print(f'amount of unique hosts: {has_host['host_id'].nunique()}')

amount of unique hosts: 3591


In [ ]:
listings['host_id'].value_counts().value_counts().sort_index()

count
1     3237
2      207
3       65
4       21
5       11
6        9
7        8
8        6
9        7
10       3
12       1
13       2
14       1
15       2
17       3
18       3
20       1
21       1
24       1
37       1
49       1
Name: count, dtype: int64

: 

In [ ]:
#How many listings each host has. 100 as highest check. (Taking anything higher will still give the same results.)

listings_per_host = pd.cut(
    has_host.groupby('host_id').size(),
    bins=[0,1,2,5,10,25,100],
    labels=['1', '2', '3-5', '6-10', '11-25', '26-100']
)

print(listings_per_host.value_counts().sort_index())

print(listings_per_host.value_counts(normalize=True).sort_index().round(3) * 100)

1         3237
2          207
3-5         97
6-10        33
11-25       15
26-100       2
Name: count, dtype: int64
1         90.1
2          5.8
3-5        2.7
6-10       0.9
11-25      0.4
26-100     0.1
Name: proportion, dtype: float64


**Verdict:** Cannot be concluded.
Like the expected answer says, it can also be argued that the argument does not hold, *because* it cannot be concluded.

None of the values we can extract about the host says anything about it being a 'professional company', which in itself is a bit of a vague phrasing.
The only thing we can tell that is somewhat related is the amount and proportion of listings per host.

## Task 5


**Note**
Of course all missing data is of big importance, just like the data that does exist. 
35% of the prices were missing, with a high correlation of these not being available at all.
14.1% of the listings/entries have no reviews, with about a third having perfect ratings. These listings do not have a particularly high amount of reviews.

https://insideairbnb.com/get-the-data/ : Shows the license being Creative Commons Attribution 4.0 International (CC BY 4.0) licence.
The snapshot, while discussed as being 26th and 27th within class, shows 25th September on the page.

Calculated columns - which came with the .csv file, and is not a product from us - are the **estimated_**somethingsomething columns.

Something this table does not tell us is first of all the whole thing about it being a "Professional company". While there are some columns for the host, this is not specified.
Also, the table does not tell us about closed listings. 

I'd want to correct claim 2 most. Even though price is valuable as well, and also missing a bigger portion, reviews have more subjectivity, which in my opinion
deserves greater attention because of more human involvement/insight.
